# Module 0: Environment Setup for Genetic Algorithms

## Learning Objectives

By completing this notebook, you will:
1. Install and configure DEAP (Distributed Evolutionary Algorithms in Python)
2. Set up required dependencies for time series forecasting
3. Verify your environment with test examples
4. Create a simple genetic algorithm from scratch
5. Run your first feature selection GA

## Prerequisites

- Python 3.8 or higher
- Basic familiarity with pip or conda
- Understanding of virtual environments (recommended)

## Estimated Time: 30-45 minutes

In [ ]:
learning_objectives(["Install and configure DEAP (Distributed Evolutionary Algorithms in Python)", "Set up required dependencies for time series forecasting", "Verify your environment with test examples", "Create a simple genetic algorithm from scratch", "Run your first feature selection GA"])

In [ ]:
section_divider("Installation")

## 1. Installation

### 1.1 Core Libraries

We'll need the following libraries:
- **DEAP**: Genetic algorithm framework
- **scikit-learn**: Machine learning models and evaluation
- **pandas**: Data manipulation
- **numpy**: Numerical computing
- **matplotlib/seaborn**: Visualization
- **statsmodels**: Time series analysis (optional but useful)

Run the cell below to install (uncomment if needed):

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

# Uncomment to install (use ! for Jupyter, % for some environments)
# !pip install deap scikit-learn pandas numpy matplotlib seaborn statsmodels

In [ ]:
apply_course_theme()
apply_plot_theme()

### 1.2 Verify Installations

Let's verify all packages are installed correctly:

In [ ]:
import sys
print(f"Python version: {sys.version}\n")

# Test imports and show versions
try:
    import deap
    print(f"✓ DEAP version: {deap.__version__}")
except ImportError as e:
    print(f"✗ DEAP not found: {e}")

try:
    import sklearn
    print(f"✓ scikit-learn version: {sklearn.__version__}")
except ImportError as e:
    print(f"✗ scikit-learn not found: {e}")

try:
    import pandas as pd
    print(f"✓ pandas version: {pd.__version__}")
except ImportError as e:
    print(f"✗ pandas not found: {e}")

try:
    import numpy as np
    print(f"✓ numpy version: {np.__version__}")
except ImportError as e:
    print(f"✗ numpy not found: {e}")

try:
    import matplotlib
    print(f"✓ matplotlib version: {matplotlib.__version__}")
except ImportError as e:
    print(f"✗ matplotlib not found: {e}")

print("\n" + "="*60)
print("If all packages show ✓, you're ready to proceed!")
print("If any show ✗, install them using the cell above.")

In [ ]:
section_divider("DEAP Basics")

## 2. DEAP Basics

### 2.1 Understanding DEAP's Architecture

DEAP (Distributed Evolutionary Algorithms in Python) uses a unique architecture:

1. **Creator**: Defines custom types (Fitness, Individual)
2. **Toolbox**: Registers functions (operators, evaluation)
3. **Algorithms**: High-level evolution strategies

Let's explore each component:

In [ ]:
callout("DEAP's three-layer architecture (Creator, Toolbox, Algorithms) cleanly separates type definitions from operator registration and execution logic.", kind="insight")

In [ ]:
from deap import base, creator, tools, algorithms
import numpy as np

# Step 1: Create custom types
# WARNING: Only run once per session (DEAP caches types)
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximize fitness
    creator.create("Individual", list, fitness=creator.FitnessMax)

print("Created custom types:")
print(f"  - FitnessMax: {creator.FitnessMax}")
print(f"  - Individual: {creator.Individual}")
print(f"\nFitness weights: {creator.FitnessMax.weights}")
print("  (positive = maximize, negative = minimize)")

### 2.2 Creating the Toolbox

The Toolbox registers all functions we'll use:

In [ ]:
# Create toolbox
toolbox = base.Toolbox()

# Register a random binary attribute generator
toolbox.register("attr_bool", np.random.randint, 0, 2)

# Register individual generator (10 binary genes)
N_FEATURES = 10
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_bool, n=N_FEATURES)

# Register population generator
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Test: Create one individual
ind = toolbox.individual()
print(f"Sample individual (binary chromosome): {ind}")
print(f"Type: {type(ind)}")
print(f"Length: {len(ind)}")

# Test: Create small population
pop = toolbox.population(n=5)
print(f"\nSample population (5 individuals):")
for i, individual in enumerate(pop):
    print(f"  Individual {i}: {individual}")

In [ ]:
section_divider("Simple GA Example: Maximize Ones")

## 3. Simple GA Example: Maximize Ones

### 3.1 Problem Definition

Before feature selection, let's solve a simple problem:

**Goal**: Find a binary string of 10 bits with maximum number of 1s

**Fitness**: Count the 1s (maximize)

This demonstrates the GA workflow:

In [ ]:
def eval_ones(individual):
    """
    Fitness function: count 1s in binary string.
    
    Parameters:
    -----------
    individual : list
        Binary chromosome [0, 1, 1, 0, ...]
    
    Returns:
    --------
    fitness : tuple
        Single-value tuple (DEAP requirement)
    """
    return (sum(individual),)  # Must return tuple!

# Test fitness function
test_ind = [1, 0, 1, 1, 0, 1, 0, 1, 1, 1]
fitness = eval_ones(test_ind)
print(f"Individual: {test_ind}")
print(f"Fitness: {fitness[0]} (sum of 1s)")

### 3.2 Register Genetic Operators

In [ ]:
# Register evaluation function
toolbox.register("evaluate", eval_ones)

# Register genetic operators
toolbox.register("mate", tools.cxTwoPoint)  # Two-point crossover
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)  # Bit-flip mutation (5% per bit)
toolbox.register("select", tools.selTournament, tournsize=3)  # Tournament selection

print("Registered operators:")
print(f"  - Evaluation: {toolbox.evaluate}")
print(f"  - Crossover: {toolbox.mate}")
print(f"  - Mutation: {toolbox.mutate}")
print(f"  - Selection: {toolbox.select}")

### 3.3 Run the Genetic Algorithm

In [ ]:
# Parameters
POPULATION_SIZE = 50
P_CROSSOVER = 0.7  # 70% probability of crossover
P_MUTATION = 0.2   # 20% probability of mutation
MAX_GENERATIONS = 40

# Create initial population
population = toolbox.population(n=POPULATION_SIZE)

# Statistics to track
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

# Hall of Fame: keep best 5 individuals
hall_of_fame = tools.HallOfFame(5)

# Run the algorithm
print("Starting evolution...\n")
population, logbook = algorithms.eaSimple(
    population, toolbox,
    cxpb=P_CROSSOVER,
    mutpb=P_MUTATION,
    ngen=MAX_GENERATIONS,
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True
)

print("\n" + "="*60)
print("Evolution completed!")
print(f"Best individual: {hall_of_fame[0]}")
print(f"Best fitness: {hall_of_fame[0].fitness.values[0]}")

### 3.4 Visualize Evolution

In [ ]:
import matplotlib.pyplot as plt

# Extract statistics
gen = logbook.select("gen")
fit_avg = logbook.select("avg")
fit_max = logbook.select("max")
fit_min = logbook.select("min")

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(gen, fit_avg, 'b-', label='Average', linewidth=2)
plt.plot(gen, fit_max, 'r-', label='Maximum', linewidth=2)
plt.plot(gen, fit_min, 'g-', label='Minimum', linewidth=2)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Fitness', fontsize=12)
plt.title('Evolution of Fitness Over Generations', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
# Show diversity (standard deviation)
fit_std = logbook.select("std")
plt.plot(gen, fit_std, 'purple', linewidth=2)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Fitness Std Dev', fontsize=12)
plt.title('Population Diversity Over Time', fontsize=14)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print(f"  - Initial avg fitness: {fit_avg[0]:.2f}")
print(f"  - Final avg fitness: {fit_avg[-1]:.2f}")
print(f"  - Improvement: {fit_avg[-1] - fit_avg[0]:.2f}")
print(f"  - Converged to optimal? {fit_max[-1] == N_FEATURES}")

In [ ]:
section_divider("Feature Selection GA Example")

In [ ]:
callout("Binary chromosome encoding maps naturally to feature selection: each bit represents whether a feature is included (1) or excluded (0).", kind="key")

## 4. Feature Selection GA Example

### 4.1 Generate Synthetic Data

Create a dataset where only some features are relevant:

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

# Generate data: 100 samples, 20 features, only 5 are informative
np.random.seed(42)
X, y = make_regression(
    n_samples=100,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    noise=10.0,
    random_state=42
)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Target variable range: [{y.min():.2f}, {y.max():.2f}]")

# Baseline: all features
model = LinearRegression()
baseline_score = cross_val_score(
    model, X, y, cv=5, scoring='neg_mean_squared_error'
).mean()
print(f"\nBaseline (all features) MSE: {-baseline_score:.4f}")

### 4.2 Define Feature Selection Fitness

In [ ]:
def eval_feature_subset(individual, X, y):
    """
    Evaluate feature subset using cross-validation.
    
    Parameters:
    -----------
    individual : list
        Binary chromosome indicating selected features
    X : array, shape (n_samples, n_features)
        Feature matrix
    y : array, shape (n_samples,)
        Target variable
    
    Returns:
    --------
    fitness : tuple
        Negative MSE (higher is better)
    """
    # Convert to boolean mask
    mask = np.array(individual, dtype=bool)
    
    # Check if at least one feature is selected
    if not np.any(mask):
        return (-1e10,)  # Very bad fitness for empty selection
    
    # Select features
    X_selected = X[:, mask]
    
    # Evaluate with cross-validation
    model = LinearRegression()
    scores = cross_val_score(
        model, X_selected, y,
        cv=5,
        scoring='neg_mean_squared_error'
    )
    
    # Return mean score (negative MSE, so higher is better)
    # Add small parsimony penalty: prefer fewer features
    parsimony_penalty = 0.01 * (np.sum(mask) / len(mask))
    
    return (scores.mean() - parsimony_penalty,)

# Test fitness function
test_individual = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
test_fitness = eval_feature_subset(test_individual, X, y)
print(f"Test individual (first 3 features): {test_individual}")
print(f"Test fitness: {test_fitness[0]:.4f}")

### 4.3 Setup GA for Feature Selection

In [ ]:
# Create new types for feature selection
if not hasattr(creator, "FitnessMaxFS"):
    creator.create("FitnessMaxFS", base.Fitness, weights=(1.0,))
    creator.create("IndividualFS", list, fitness=creator.FitnessMaxFS)

# New toolbox
toolbox_fs = base.Toolbox()

# Register generators (20 features now)
toolbox_fs.register("attr_bool", np.random.randint, 0, 2)
toolbox_fs.register(
    "individual",
    tools.initRepeat,
    creator.IndividualFS,
    toolbox_fs.attr_bool,
    n=X.shape[1]  # Number of features
)
toolbox_fs.register("population", tools.initRepeat, list, toolbox_fs.individual)

# Register operators
toolbox_fs.register("evaluate", eval_feature_subset, X=X, y=y)
toolbox_fs.register("mate", tools.cxTwoPoint)
toolbox_fs.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox_fs.register("select", tools.selTournament, tournsize=3)

print("Feature Selection GA configured")
print(f"Chromosome length: {X.shape[1]} (one bit per feature)")

### 4.4 Run Feature Selection GA

In [ ]:
# Create population
pop_fs = toolbox_fs.population(n=50)

# Statistics
stats_fs = tools.Statistics(lambda ind: ind.fitness.values)
stats_fs.register("avg", np.mean)
stats_fs.register("max", np.max)

# Hall of Fame
hof_fs = tools.HallOfFame(5)

# Run evolution
print("Starting feature selection GA...\n")
pop_fs, log_fs = algorithms.eaSimple(
    pop_fs, toolbox_fs,
    cxpb=0.7,
    mutpb=0.2,
    ngen=30,
    stats=stats_fs,
    halloffame=hof_fs,
    verbose=True
)

print("\n" + "="*60)
print("Feature selection completed!")
print(f"\nBest solution:")
best = hof_fs[0]
selected_features = [i for i, bit in enumerate(best) if bit == 1]
print(f"  Selected features: {selected_features}")
print(f"  Number of features: {len(selected_features)}/{len(best)}")
print(f"  Fitness: {best.fitness.values[0]:.4f}")
print(f"  Baseline fitness: {baseline_score:.4f}")

### 4.5 Visualize Results

In [ ]:
# Plot evolution
gen_fs = log_fs.select("gen")
avg_fs = log_fs.select("avg")
max_fs = log_fs.select("max")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(gen_fs, avg_fs, 'b-', label='Average Fitness', linewidth=2)
plt.plot(gen_fs, max_fs, 'r-', label='Best Fitness', linewidth=2)
plt.axhline(y=baseline_score, color='green', linestyle='--',
            label='Baseline (all features)', linewidth=2)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Fitness (Negative MSE)', fontsize=12)
plt.title('Feature Selection Evolution', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
# Show selected features
plt.bar(range(len(best)), best, color='steelblue', edgecolor='black')
plt.xlabel('Feature Index', fontsize=12)
plt.ylabel('Selected (1) or Not (0)', fontsize=12)
plt.title('Best Feature Subset', fontsize=14)
plt.ylim(-0.1, 1.1)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
section_divider("Exercises")

## 5. Exercises

### Exercise 5.1: Experiment with Parameters

Modify the GA parameters and observe the effects:

**Task**: Run the feature selection GA with different mutation rates and compare results.

In [ ]:
# YOUR CODE HERE
# ---------------
# Try mutation rates: 0.01, 0.1, 0.5
# Compare:
#   - Final best fitness
#   - Number of features selected
#   - Convergence speed



### Exercise 5.2: Custom Fitness Function

**Task**: Modify the fitness function to penalize feature subsets with more than 10 features.

In [ ]:
# YOUR CODE HERE
# ---------------
def eval_feature_subset_constrained(individual, X, y, max_features=10):
    """
    Feature selection with hard constraint on number of features.
    
    Hint: Add a large penalty if sum(individual) > max_features
    """
    # TODO: Implement
    pass

# Test your function
# test_ind = [1]*15 + [0]*5
# fitness = eval_feature_subset_constrained(test_ind, X, y, max_features=10)
# print(f"Fitness with 15 features (limit=10): {fitness}")

### Exercise 5.3: Compare with Filter Method

**Task**: Compare GA-selected features with a filter method (mutual information).

In [ ]:
# YOUR CODE HERE
# ---------------
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# Select top k features using MI
k = len(selected_features)  # Same number as GA selected
# TODO: Use SelectKBest to select features
# TODO: Compare MSE of GA vs MI features



In [ ]:
section_divider("Summary")

## 6. Summary

### Key Takeaways

1. **DEAP Architecture**: Creator → Toolbox → Algorithms
2. **GA Components**: Population, fitness evaluation, selection, crossover, mutation
3. **Feature Selection**: Binary encoding, one bit per feature
4. **Fitness Design**: Balance accuracy and parsimony
5. **Evolution Monitoring**: Track statistics and visualize progress

### What's Next

In Module 1, we'll dive deeper into:
- Chromosome encoding strategies
- Selection operators (tournament, roulette, rank)
- Crossover operators (single-point, two-point, uniform)
- Mutation strategies
- Building GAs from scratch (no DEAP)

### Additional Resources

- **DEAP Documentation**: https://deap.readthedocs.io/
- **DEAP Examples**: https://github.com/DEAP/deap/tree/master/examples
- **GA Tutorial**: "A Field Guide to Genetic Programming" (free PDF)

### Environment Check

Run the cell below to verify your setup is complete:

In [ ]:
key_takeaways(["Creator → Toolbox → Algorithms", "Population, fitness evaluation, selection, crossover, mutation", "Binary encoding, one bit per feature", "Balance accuracy and parsimony", "Track statistics and visualize progress"])

In [ ]:
print("Environment Setup - Final Check")
print("="*60)

checks = {
    "DEAP installed": 'deap' in sys.modules,
    "scikit-learn installed": 'sklearn' in sys.modules,
    "Can create GA types": hasattr(creator, "FitnessMax"),
    "Can run algorithms": 'algorithms' in dir(),
    "Can visualize": 'plt' in dir(),
}

all_pass = True
for check, result in checks.items():
    status = "✓" if result else "✗"
    print(f"{status} {check}")
    if not result:
        all_pass = False

print("\n" + "="*60)
if all_pass:
    print("✓ All checks passed! You're ready for Module 1.")
else:
    print("✗ Some checks failed. Please review the installation steps.")